# 01 — Análisis Exploratorio de Datos (EDA)

**Observatorio de Ciencia, Tecnología e Innovación — Grupo 7**

Este notebook realiza el análisis exploratorio del dataset consolidado de
investigadores reconocidos por Minciencias — **6 convocatorias (2013–2021)**,
77.237 registros y 30 variables.

**Contenido:**
1. Carga y vista general de los datos
2. Calidad de datos (valores faltantes, tipos)
3. Variables numéricas — estadísticas descriptivas
4. Variables categóricas — frecuencias
5. Visualizaciones clave

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path().resolve().parent  # raiz del repo (notebooks/../)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ingesta import cargar_consolidado
from Transformacion import transformar

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Carga de datos

In [ ]:
df_raw = cargar_consolidado()
df = transformar(df_raw)
print(f'Shape: {df.shape}')
df.head()

## 2. Calidad de datos

In [ ]:
# Tipos de datos
print('Tipos de datos:')
print(df.dtypes)

# Valores faltantes
faltantes = df.isnull().mean().sort_values(ascending=False) * 100
print('\nPorcentaje de valores faltantes:')
print(faltantes[faltantes > 0])

In [ ]:
# Mapa de calor de valores faltantes
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    df.isnull().T,
    cbar=False,
    ax=ax,
    yticklabels=True,
    xticklabels=False,
    cmap='Blues'
)
ax.set_title('Mapa de valores faltantes por columna')
plt.tight_layout()
plt.show()

## 3. Variables numéricas

In [ ]:
num_vars = ['EDAD_ANOS_PR', 'NRO_ORDEN_FORM_PR', 'ORDEN_CLAS_PR']
df[num_vars].describe().T

In [ ]:
fig, axes = plt.subplots(1, len(num_vars), figsize=(15, 4))
for ax, col in zip(axes, num_vars):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribución de {col}')
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

## 4. Variables categóricas clave

In [ ]:
cat_cols = [
    'NME_GENERO_PR',
    'NME_GRAN_AREA_PR',
    'NME_NIV_FORM_PR',
    'NME_CLASIFICACION_PR',
    'NME_REGION_RES_PR',
]

for col in cat_cols:
    if col in df.columns:
        print(f'\n===== {col} =====')
        print(df[col].value_counts(dropna=False).head(10))

## 5. Visualizaciones

In [ ]:
# Distribución por género
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genero_cnt = df['NME_GENERO_PR'].value_counts()
axes[0].pie(
    genero_cnt.values,
    labels=genero_cnt.index,
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452', '#55A868']
)
axes[0].set_title('Distribución por género')

# Distribución por gran área
area_cnt = df['NME_GRAN_AREA_PR'].value_counts().head(8)
axes[1].barh(area_cnt.index, area_cnt.values, color='steelblue')
axes[1].set_title('Top 8 grandes áreas OCDE')
axes[1].set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de edad por convocatoria
if 'ANO_CONVO_INT' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=df.dropna(subset=['ANO_CONVO_INT', 'EDAD_ANOS_PR']),
        x='ANO_CONVO_INT',
        y='EDAD_ANOS_PR',
        palette='Blues'
    )
    plt.title('Distribución de edad por convocatoria')
    plt.xlabel('Año de convocatoria')
    plt.ylabel('Edad (años)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 15 departamentos de residencia
top_dep = df['NME_DEPARTAMENTO_RES_PR'].value_counts().head(15)
plt.figure(figsize=(10, 6))
sns.barplot(y=top_dep.index, x=top_dep.values, palette='Blues_r')
plt.title('Top 15 departamentos de residencia')
plt.xlabel('Número de investigadores')
plt.tight_layout()
plt.show()